# 2D Arrays & Matrices in Python (Lists of Lists)

A 2D array is just a list where each element is itself a list. But the patterns, pitfalls, and power multiply. Let’s go deep.

---

### 1. What Is a 2D Array?

In [ ]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

**Visually**:

|         | Col 0  |  Col 1  |  Col 2  |
|---------|--------|---------|---------|
| Row 0   |   1    |    2    |    3    |
| Row 1   |   4    |    5    |    6    |
| Row 2   |   7    |    8    |    9    |

**In memory**: It is NOT a contiguous grid. It is a list of references, where each reference points to another list.

```txt
matrix ──→ [ref0, ref1, ref2]
            ↓      ↓      ↓
           [1,2,3] [4,5,6] [7,8,9]
```

This distinction matters for the *`[[0]*3]*3`* trap and for deep copying.

> For more detail to know about 2d matrix representation, please read [1d_2d_memory_representation](./interview/1d_2d_list_memory_representation.md)

---

### 2. Creating 2D Arrays (Every Way)

**2.1 Row by Row (Explicit)**

In [1]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

**2.2 Pre-allocated with Zeros (The RIGHT Way)**

In [ ]:
rows, cols = 3, 4

# CORRECT: Each row is a NEW list object
matrix = [[0 for _ in range(cols)] for _ in range(rows)]

# Syntax breakdown
"""
matrix = []
# 1. This runs FIRST (corresponds to the right side)
for _ in range(rows):
    row = []
    # 2. This runs SECOND (corresponds to the left side)
    for _ in range(cols):
        row.append(0)
    matrix.append(row)
"""

# OR using multiplication for the INNER list only
matrix = [[0] * 3 for _ in range(rows)]

"""
Both produce:
[[0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0]]
"""

```txt
matrix = [ [0 for _ in range(cols)] for _ in range(rows) ]
           └───────── 2 ─────────┘ └──────── 1 ────────┘
             Inner loop runs next.   Outer loop starts here.
             Creates column items    Determines the total
             for each single row.    number of rows.
```

**2.3 The Catastrophic Trap (WRONG Way)**

In [ ]:
rows, cols = 3, 4

# WRONG: All rows point to the SAME inner list!
matrix = [[0] * cols] * rows

matrix[0][0] = 99
print(matrix)

# [[99, 0, 0, 0],
#  [99, 0, 0, 0],   ← Every row changed!
#  [99, 0, 0, 0]]

> Why? * on the outer list copies the **reference** to the inner list. Three rows, one list.

Verify with id():

In [4]:
bad = [[0]*3]*3
print(id(bad[0]) == id(bad[1]))   # True ← Same object!

good = [[0]*3 for _ in range(3)]
print(id(good[0]) == id(good[1])) # False ← Different objects!

True
False


**2.4 From User Input**

In [5]:
rows = int(input("Rows: "))
cols = int(input("Cols: "))

matrix = []

for i in range(rows):
    row = list(map(int, input(f"Row {i}: ").split()))
    matrix.append(row)
print(matrix)

[[3], [2], [2]]


In [6]:
# Or compact

matrix = [list(map(int, input().split())) for _ in range(rows)]

print(matrix)

[[5], [1], [2]]


**2.5 From a Flat List**

In [7]:
flat = [1, 2, 3, 4, 5, 6, 7, 8, 9]
rows, cols = 3, 3

matrix = [flat[i*cols:(i+1)*cols] for i in range(rows)]

print(matrix)

[[1, 2, 3], [4, 5, 6], [7, 8, 9]]


**2.6 Identity Matrix**

In [ ]:
n = 4

identity = [[1 if i == j else 0 for j in range(n)] for i in range(n)]

print(identity)

"""
# [[1, 0, 0, 0],
#  [0, 1, 0, 0],
#  [0, 0, 1, 0],
#  [0, 0, 0, 1]]
"""

**2.7 Random Matrix**

In [ ]:
import random

rows, cols = 3, 4

matrix = [[random.randint(1, 100) for _ in range(cols)] for _ in range(rows)]

"""
[[69 25 18 96]
 [20 58 29 94]
 [46 31 29 99]]
"""

---

### 3. Indexing & Accessing Elements

**3.1 Single Element**

In [ ]:
matrix = [
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
]

matrix[0][0]        # 10 (Row 0, Col 0)
matrix[1][2]        # 50 (Row 1, Col 2)
matrix[2][1]        # 80 (Row 2, Col 3)

80

**Step-by-step evaluation:**

In [15]:
# get full row
print(matrix[1])

# from that row get the specific element at specified index
print(matrix[1][2])

[40, 50, 60]
60


**3.2 Negative Indices**

In [ ]:
print(matrix[-1])           # last row [70, 80, 90]
print(matrix[-1][-1])       # first [-1] return last row and second gives us last element in that list (90  (last row, last col))

[70, 80, 90]
90


In [17]:
matrix[-1][0]    # 70  (last row, first col)

70

**3.3 Entire Row or Column**

In [ ]:
# Entire row (easy — it's just a list)
row1 = matrix[1]                        # [40, 50, 60]

# Entire column (must extract)
col1 = [row[1] for row in matrix]       # [20, 50, 80]

**3.4 Bounds Checking**

In [18]:
rows = len(matrix)
cols = len(matrix[0])

# Safe access pattern
r, c = 5, 5

if 0 <= r < rows and 0 <= c < cols:
    print(matrix[r][c])
else:
    print("Out of bounds")

Out of bounds


---

### 4. Traversing a 2D Array (All Patterns)

In [20]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

**4.1 Row-Major Order (Left → Right, Top → Bottom)**

In [24]:
# Standard nested loop
for i in range(len(matrix)):
    for j in range(len(matrix[0])):
        print(matrix[i][j], end=' ')
    print()

1 2 3 
4 5 6 
7 8 9 


**4.2 Column-Major Order (Top → Bottom, Left → Right)**

In [27]:
for i in range(len(matrix)):
    for j in range(len(matrix[0])):
        print(matrix[j][i], end=' ')        # just swap matrix[i][j] -> matrix[j][i]
    print()


# separator
print("-"*4)

# Alternative solution

# get columns size first (width)
for j in range(len(matrix[0])):
    # get matrix height (row size)
    for i in range(len(matrix)):
        print(matrix[i][j], end=' ')
    print()

1 4 7 
2 5 8 
3 6 9 
----
1 4 7 
2 5 8 
3 6 9 


**4.3 With enumerate (Index + Value)**

In [28]:
for i, row in enumerate(matrix):
    for j, val in enumerate(row):
        print(f"matrix[{i}][{j}] = {val}")

matrix[0][0] = 1
matrix[0][1] = 2
matrix[0][2] = 3
matrix[1][0] = 4
matrix[1][1] = 5
matrix[1][2] = 6
matrix[2][0] = 7
matrix[2][1] = 8
matrix[2][2] = 9


**4.4 Direct Row Iteration (When indices don't matter)**

In [29]:
for row in matrix:
    for val in row:
        print(val, end=" ")
    print()

1 2 3 
4 5 6 
7 8 9 


**4.5 Flattening (1D from 2D)**

In [30]:
# Method 1: Nested loop
flat = []
for row in matrix:
    for val in row:
        flat.append(val)

print(flat)

[1, 2, 3, 4, 5, 6, 7, 8, 9]


In [31]:
# Method 2: List comprehension (the pythonic way)
flat = [val for row in matrix for val in row]   # Read left-to-right like nested loops!
print(flat)

[1, 2, 3, 4, 5, 6, 7, 8, 9]


In [33]:
# Method 3: sum() with empty start (clever but O(n²) — avoid for big data)
flat = sum(matrix, [])
print(flat)         
# [[1,2,3]] + [] = [1,2,3], then + [4,5,6]...

[1, 2, 3, 4, 5, 6, 7, 8, 9]


### 6.1 Main Diagonal (Top-Left to Bottom-Right)

**6.1 Main Diagonal (Top-Left to Bottom-Right)**

In [34]:
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

In [37]:
# Diagonals
diag = [matrix[i][i] for i in range(len(matrix))]
print(diag)

[1, 5, 9]


**6.2 Anti-Diagonal (Top-Right to Bottom-Left)**

In [ ]:
# Anti-Diagonals
anti_diag = [matrix[i][n-1-i] for i in range(len(matrix))]
print(anti_diag)

[3, 5, 7]
